In [0]:
%sql
select * from pysparkcatalogue.source.products;

id,name,price,category,updateddate
1,iphone,1000,electronics,2026-05-19T21:07:51.871Z
2,Macbook,2000,electronics,2026-05-19T21:07:51.871Z
3,T-Shirt,300,clothing,2026-05-19T21:07:51.871Z
4,Shirt,100,clothing,2026-05-19T21:07:51.871Z
5,Pants,150,clothing,2026-05-19T21:07:51.871Z
5,Trouser,150,clothing,2026-05-19T21:12:32.583Z
4,Shirt,250,clothing,2026-05-19T21:24:37.155Z


In [0]:
# source
df= spark.read.table("pysparkcatalogue.source.products")
df.createOrReplaceTempView("abc")

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc ,col
df = df.withColumn("dedup", row_number().over(Window.partitionBy('id').orderBy(desc('updatedDate'))))

df =df.filter(col("dedup")==1).drop('dedup')
display(df)

id,name,price,category,updateddate
1,iphone,1000,electronics,2026-05-19T21:07:51.871Z
2,Macbook,2000,electronics,2026-05-19T21:07:51.871Z
3,T-Shirt,300,clothing,2026-05-19T21:07:51.871Z
4,Shirt,250,clothing,2026-05-19T21:24:37.155Z
5,Trouser,150,clothing,2026-05-19T21:12:32.583Z


In [0]:
#volume 
# pysparkcatalogue.source.products; inorder to do Upsert we need to store in some destination and then do upsert
# sink folder is the destination called volumne dbvolume in pysparkcatalogue--sourceschema--volume
#  /Volumes/pysparkcatalogue/source/dbvolume/products_sink/

# creating delta object 

from delta.tables import DeltaTable
#create detla object on sinklocation

if len(dbutils.fs.ls("/Volumes/pysparkcatalogue/source/dbvolume/products_sink/"))>0:
    dltobj = DeltaTable.forPath(spark, "/Volumes/pysparkcatalogue/source/dbvolume/products_sink/")
        #add condition so that it will compare based on dates
    dltobj.alias("trg").merge(
            df.alias("src"),
            "src.id = trg.id")\
            .whenMatchedUpdateAll(condition="src.updateddate >= trg.updateddate")\
            .whenNotMatchedInsertAll() \
            .execute()
    print("This is upserting now")

else:
    df.write.format("delta")\
            .mode("overwrite")\
            .save("/Volumes/pysparkcatalogue/source/dbvolume/products_sink/")

This is upserting now


In [0]:
%sql
select * from delta.`/Volumes/pysparkcatalogue/source/dbvolume/products_sink/`


id,name,price,category,updateddate
1,iphone,1000,electronics,2026-05-19T21:07:51.871Z
2,Macbook,2000,electronics,2026-05-19T21:07:51.871Z
3,T-Shirt,300,clothing,2026-05-19T21:07:51.871Z
4,Shirt,250,clothing,2026-05-19T21:24:37.155Z
5,Trouser,150,clothing,2026-05-19T21:12:32.583Z
5,Trouser,150,clothing,2026-05-19T21:12:32.583Z


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
class Datavalidation:

    def __init__(self,df):
        self.df = df
 
    def dedup(self,keyCol, cdcCol):
        df = self.df.withColumn("dedup", row_number().over(Window.partitionBy(keyCol).orderBy(desc(cdcCol))))
        df = df.filter(col('dedup')==1).drop('dedup')
        return df
    
    def removeNulls(self,nullCol):
        df = self.df.filter(col(nullCol).isNotNull())
        return df

In [0]:
df =spark.createDataFrame([("1","2020-01-01",None),("2","2021-01-01",200),("3","2021-01-01",300),("1","2020-01-01",100)],["order_id","order_date","amount"])

display(df)

order_id,order_date,amount
1,2020-01-01,null
2,2021-01-01,200
3,2021-01-01,300
1,2020-01-01,100


In [0]:
#@instance

obj1 =Datavalidation(df)

df_dedup = obj1.dedup("order_id","order_date")

display(df_dedup)


order_id,order_date,amount
1,2020-01-01,
2,2021-01-01,200
3,2021-01-01,300


In [0]:
obj1 =Datavalidation(df)

a = obj1.removeNulls(nullCol='amount')

display(a)


order_id,order_date,amount
2,2021-01-01,200
3,2021-01-01,300
1,2020-01-01,100
